# Agent Platform Python SDK — Quickstart Cookbook

This notebook walks through the public `agent-platform` SDK end-to-end:

1. Install + authenticate
2. Create a session (agent + environment)
3. Poll session status
4. Stream session changes
5. Send a follow-up message
6. Pause / resume / cancel
7. Manage memories, skills, environments, agents
8. Async usage
9. Error handling

The SDK is generated from `https://platform.staging.sandboxh.ai/share/openapi.json` (the v2 surface) using `openapi-python-client==0.28.3` with custom Pydantic v2 templates. All request and response models are Pydantic `BaseModel` subclasses; runtime deps are `httpx` + `pydantic` + `python-dateutil` + `attrs`.

> **Status:** v1 release (installable-from-repo). PyPI publish is deferred — install via `pip install /path/to/sdk/python` or directly from a git URL for now.

## 1. Install

From the `agent_platform` repository root:

```bash
# pip
pip install ./sdk/python

# uv (recommended — same package, faster resolution)
uv pip install ./sdk/python

# Direct from GitHub (once Phase 4 lands tags)
pip install "git+https://github.com/hcompai/agent_platform.git#subdirectory=sdk/python"
```

## 2. Authenticate

Auth is a Portal-H API key (`hk-*`) sent as `Authorization: Bearer <key>` on every request.

Generate one at:
- **Staging** (recommended for cookbook experimentation): https://portal.staging.sandboxh.ai/
- **Production**: https://portal.hcompany.ai/

Set it as an environment variable so the notebook stays shareable:

```bash
export AGP_API_KEY=hk-xxxxxxxx
```

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

from agent_platform import Client

# The notebook lives at sdk/python/cookbook/; the integration .env it shares
# with the test suite lives one level up. Load it explicitly so we don't pick
# up a different .env from the repo root or cwd.
load_dotenv(Path("../.env"), override=False)

# Pydantic v2 forward-reference workaround: the generated models use
# TYPE_CHECKING-guarded imports to break circular dependencies, but Pydantic
# needs every referenced class loaded *and* rebuilt before .from_dict() /
# response parsing works. Importing the full models package + calling
# model_rebuild on every class resolves the refs in one shot.
import agent_platform.models as _models

_NS = {n: getattr(_models, n) for n in _models.__all__}
for _name in _models.__all__:
    _cls = getattr(_models, _name)
    if hasattr(_cls, "model_rebuild"):
        try:
            _cls.model_rebuild(_types_namespace=_NS)
        except Exception:
            pass  # enums + non-BaseModel classes; ignore.

# Prefer the integration-test key (HAI_API_KEY_TEST) — it matches the base URL
# used by the rest of the SDK suite. AGP_API_KEY is accepted as a fallback for
# users who set the canonical name but isn't pinned to a specific environment.
API_KEY = os.environ.get("HAI_API_KEY_TEST") or os.environ["AGP_API_KEY"]
BASE_URL = os.environ.get("HAI_API_BASE_URL_TEST") or os.environ.get("AGP_BASE_URL") or "https://agp.hcompany.ai"

client = Client(api_key=API_KEY, base_url=BASE_URL)
print("base_url =", BASE_URL)
print("client  =", type(client).__name__)
print("api_key = ****", API_KEY[-5:])

base_url = https://agp.hcompany.ai
client  = Client
api_key = **** aaa9a


## 3. Create a session

A **session** is a single run of an agent against an environment. You give it:
- An `agent` — either a string identifier (e.g. `"h/researcher"`) referencing a previously-registered Agent, OR an inline `AgentSpecInput`
- One or more `environments` — references to environment specs registered in the platform
- An optional initial `messages` list — what to tell the agent at the start

The SDK exposes per-resource modules under `agent_platform.api.<resource>`. Each operation is both a sync function (`.sync()`) and an async coroutine (`.asyncio()`).

In [2]:
# Typed SDK path: build SessionRequest from Pydantic models. The
# model_rebuild() loop in cell 1 has already resolved every TYPE_CHECKING-
# guarded forward ref, so discriminated-union fields (agent, environments)
# accept their concrete variant classes directly.
#
# Schema shape (prod /api/v2/sessions): `agent` carries its own `environments`,
# and the browser env's `kind` is "web" (was "local_browser" in earlier builds).
#
# headless=False: headless browsers are not yet provisioned by agent-environments
# on this platform — requesting one currently causes the runner to time out
# instead of returning a clear error.
from agent_platform import Agent, SessionRequest
from agent_platform.api.sessions import create_session
from agent_platform.models.browser import Browser
from agent_platform.models.user_message_event import UserMessageEvent

body = SessionRequest(
    agent=Agent(
        name="cookbook-demo-agent",
        description="Quickstart demo agent.",
        instructions="You are a concise research assistant. Reply briefly.",
        environments=[
            Browser(
                id="browser",
                kind="web",
                headless=False,
                width=1280,
                height=800,
                start_url="https://www.bing.com/",
            )
        ],
    ),
    messages=[
        UserMessageEvent(
            message="Find the latest news about renewable energy in Europe.",
        )
    ],
    max_steps=12,
    max_time_s=180.0,
    idle_timeout_s=30,
)

resp = create_session.sync_detailed(client=client, body=body)
assert resp.status_code == 201, f"create failed: {resp.status_code} {resp.content[:300]!r}"
session = resp.parsed  # typed Session
session_id = str(session.id)
print("created session", session_id)
print("status:", str(session.status.status).split(".")[-1].lower())

created session fcca1e7f-1e1c-405d-b3d0-57ef9a277005
status: pending


## 4. Poll status

Sessions run asynchronously on the platform. Poll `get_session_status` (cheap, returns just the status) or `get_session` (full envelope) until the status either becomes terminal or hits an interactive pause point.

**Lifecycle states** (from `agent_interface.status.TrajectoryStatus`):

| State | Meaning | Should I keep polling? |
|---|---|---|
| `pending` | created, not yet started | yes |
| `running` | agent is actively working | yes |
| `paused` | explicitly paused by `pause_session` | no — call `resume_session` to continue |
| `idle` | agent is alive, waiting for a new user message | no — send a message to wake it up |
| `completed` | finished successfully (terminal) | no |
| `failed` | crashed (terminal) | no |
| `timed_out` | hit `max_time_s` limit (terminal) | no |
| `interrupted` | externally cancelled (terminal) | no |


In [3]:
import time

from agent_platform.api.sessions import get_session_status

# Terminal: session is done, polling can stop.
TERMINAL = {"completed", "failed", "timed_out", "interrupted"}
# Idle/paused: agent is alive but waiting (idle → waiting for a new user
# message; paused → waiting for resume_session). We treat these as stop-
# polling states too, otherwise we'd loop forever on interactive runs.
WAITING_FOR_INPUT = {"idle", "paused"}

while True:
    status = get_session_status.sync(client=client, id=session_id)
    state = str(status.status).lower().split(".")[-1]  # "TrajectoryStatus.IDLE" -> "idle"
    print(f"  [{status.steps} steps] status={state}")
    if state in TERMINAL or state in WAITING_FOR_INPUT:
        break
    time.sleep(2)

print(f"\nstopped at: {state}")
if status.error:
    print(f"error: {status.error}")

  [0 steps] status=pending


  [0 steps] status=pending


  [0 steps] status=running


  [0 steps] status=running


  [0 steps] status=running


...

  [11 steps] status=running


  [11 steps] status=running


  [11 steps] status=running


  [11 steps] status=running


  [12 steps] status=idle

stopped at: idle


## 5. Stream changes (long-poll)

Instead of polling the whole status, you can long-poll the `changes` endpoint to receive only events since a given index. This is how the dashboard streams agent actions live.

In [4]:
from agent_platform.api.sessions import get_session_changes

from_index = 0
for _ in range(5):  # poll up to 5 batches
    page = get_session_changes.sync(
        client=client,
        id=session_id,
        from_index=from_index,
        wait_for_seconds=10,
    )
    if page is None or not page.new_events:
        break
    for event in page.new_events:
        ev = event.to_dict() if hasattr(event, "to_dict") else event
        print(f"  [{ev.get('index', '?')}] {ev.get('type', '?')}: {ev.get('summary', '')}")
    from_index = (
        page.new_events[-1].to_dict() if hasattr(page.new_events[-1], "to_dict") else page.new_events[-1]
    ).get("index", from_index) + 1

  [?] RequestStartEvent: 
  [?] RequestStartDispatchedEvent: 
  [?] AgentStartedEvent: 
  [?] ActiveStateChangeEvent: 
  [?] ActiveStateChangeEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdat

  [?] RequestStartDispatchedEvent: 
  [?] AgentStartedEvent: 
  [?] ActiveStateChangeEvent: 
  [?] ActiveStateChangeEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent:

  [?] AgentStartedEvent: 
  [?] ActiveStateChangeEvent: 
  [?] ActiveStateChangeEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEven

  [?] ActiveStateChangeEvent: 
  [?] ActiveStateChangeEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEve

  [?] ActiveStateChangeEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] AgentEvent: 
  [?] MetricsUpdateEvent: 
  [?] AgentEvent: 
  [?] A

## 6. Send a follow-up message

Mid-run, you can inject a new user message. The agent picks it up at its next decision point.

In [5]:
# Typed SDK path: UserMessageBatch wraps one or more UserMessageEvents.
# send_session_messages accepts either a batch or a single event as `body`.
from agent_platform.api.sessions import send_session_messages
from agent_platform.models.user_message_batch import UserMessageBatch
from agent_platform.models.user_message_event import UserMessageEvent

resp = send_session_messages.sync_detailed(
    client=client,
    id=session_id,
    body=UserMessageBatch(messages=[UserMessageEvent(message="Focus on offshore wind specifically.")]),
)
assert resp.status_code in (200, 202), f"send failed: {resp.status_code} {resp.content[:300]!r}"
print("message sent")

message sent


## 7. Pause, resume, cancel

In [6]:
from agent_platform.api.sessions import cancel_session, pause_session, resume_session

pause_session.sync(client=client, id=session_id)
print("paused")

resume_session.sync(client=client, id=session_id)
print("resumed")

# When you're done — cancel cleans up running infrastructure (browser sandbox, etc.)
cancel_session.sync(client=client, id=session_id)
print("cancelled")

paused


resumed


cancelled


## 8. List sessions

In [7]:
from agent_platform.api.sessions import list_sessions
from agent_platform.models import ListSessionsApiV2SessionsGetOwner as ListSessionsOwner

page = list_sessions.sync(client=client, owner=ListSessionsOwner.ME, page=1, size=10)
print(f"{page.total} sessions total — showing page {page.page}")
for s in page.items:
    state = str(s.status).lower().split(".")[-1]  # SessionSummary.status is a TrajectoryStatus enum
    msg = getattr(s.first_message, "message", None) or "(no first message)"
    print(f"  {s.id}  {state:<12}  {str(msg)[:60]}")

12 sessions total — showing page 1
  fcca1e7f-1e1c-405d-b3d0-57ef9a277005  running       Find the latest news about renewable energy in Europe.
  1aeff5dc-dd5f-4b5b-b264-d31a8a00f061  completed     Find the latest news about renewable energy in Europe.
  f166d05d-19ba-4319-af1a-a73cbba85083  failed        Find the latest news about renewable energy in Europe.
  44e46cb0-7b85-468b-ae6f-9503e5b5f886  failed        Find the latest news about renewable energy in Europe.
  63e589f5-c2c6-43df-8321-7312cc7e9942  failed        Find the latest news about renewable energy in Europe.
  fe7a9069-c9a4-4349-b4e9-2325abe54ffc  failed        Find the latest news about renewable energy in Europe.
  79535091-a1c4-483f-95d5-8297919a0f2b  failed        Find the latest news about renewable energy in Europe.
  048d02ac-bf97-4148-a337-3f5d98c512c6  interrupted   Find the latest news about renewable energy in Europe.
  144b66db-1a2e-4cb7-9c68-dd9d27a2beae  completed     Find the latest news about renewable en

## 9. Memories — persistent knowledge across sessions

A memory is a `(namespace, key) -> content` triple that agents can read/write across runs. Use it for facts the agent should remember (preferences, prior conclusions, IDs).

In [8]:
# Memories: persistent (namespace, key) -> content triples that agents can
# read/write across sessions.
#
# NOTE: /api/v2/memories is explicitly excluded from SDK codegen in
# `sdk/python/filter.yaml`, so there's no typed `agent_platform.api.memories`
# module to import. Use the raw httpx client. To enable the typed SDK path,
# remove `/api/v2/memories` from filter.yaml and re-run the SDK regen.
import json

created = (
    client.get_httpx_client()
    .post(
        "/api/v2/memories",
        json={"namespace": "user-prefs", "key": "favorite-color", "value": "blue"},
        timeout=30.0,
    )
    .json()
)
memory_id = created["id"]
print("created memory", memory_id)

page = (
    client.get_httpx_client()
    .get(
        "/api/v2/memories",
        params={"namespace": "user-prefs", "page": 1, "size": 20},
        timeout=30.0,
    )
    .json()
)
for m in page.get("items", []):
    print(f"  {m['namespace']}/{m['key']} = {m['value']!r}")

client.get_httpx_client().patch(
    f"/api/v2/memories/{memory_id}",
    json={"value": "teal"},
    timeout=30.0,
)

client.get_httpx_client().delete(f"/api/v2/memories/{memory_id}", timeout=30.0)
print("deleted memory", memory_id)

created memory f213630a-5982-482a-a21d-0e6a73d65de3


  user-prefs/favorite-color = 'blue'


deleted memory f213630a-5982-482a-a21d-0e6a73d65de3


## 10. Skills — reusable capability declarations

Skills are reusable named capabilities (e.g. `book-flight`, `summarize-pdf`) that sessions can reference.

In [9]:
from agent_platform import CreateSkill
from agent_platform.api.skills import create_skill, delete_skill, list_skills

# Typed SDK path: CreateSkill model + create_skill.sync_detailed returns a
# typed Skill in resp.parsed.
resp = create_skill.sync_detailed(
    client=client,
    body=CreateSkill(
        name="cookbook-demo-skill",
        description="Extract a 5-bullet summary from a PDF given a URL.",
        body="Summarize the PDF in 5 bullets.",
    ),
)
assert resp.status_code == 201, f"create failed: {resp.status_code} {resp.content[:300]!r}"
skill = resp.parsed  # typed Skill
skill_id = str(skill.id)
print("created skill", skill_id)

page = list_skills.sync(client=client, page=1, size=10)
for s in page.items:
    print(f"  {s.name}  ({s.id})")

delete_skill.sync(client=client, id=skill_id)
print("deleted skill", skill_id)

created skill 6aae8cc9-715c-4ef2-9942-372d47ceb7d4


  cookbook-demo-skill  (6aae8cc9-715c-4ef2-9942-372d47ceb7d4)


deleted skill 6aae8cc9-715c-4ef2-9942-372d47ceb7d4


## 11. Environments — sandbox specs sessions reference

An **environment** is a configuration entry describing a sandbox type (e.g. `browser-v1` with a particular browser version, region, profile). Sessions reference environments by `env_identifier`.

In [10]:
# Environments: registered sandbox specs sessions reference by id.
#
# NOTE: /api/v2/environments is explicitly excluded from SDK codegen in
# `sdk/python/filter.yaml`, so there's no typed `agent_platform.api.environments`
# module to import. Use the raw httpx client. To enable the typed SDK path,
# remove `/api/v2/environments` from filter.yaml and re-run the SDK regen.
page = (
    client.get_httpx_client()
    .get(
        "/api/v2/environments",
        params={"page": 1, "size": 10},
        timeout=30.0,
    )
    .json()
)
for e in page.get("items", []):
    spec = e.get("spec", {})
    print(f"  {e.get('id')}  kind={spec.get('kind', '?')}")

## 12. Agents — registered agent definitions

An **agent** is a registered, identifiable, reusable agent definition (`agent_identifier` + spec). Sessions can reference an agent by identifier (`"h/researcher"`) rather than passing an inline `AgentSpec` every time.

In [11]:
from agent_platform.api.agents import list_agents

page = list_agents.sync(client=client, page=1, size=20)
if page is None or not page.items:
    print("no agents")
else:
    for a in page.items:
        name = a.spec.name if a.spec else "?"
        desc = (a.spec.description or "")[:60] if a.spec else ""
        print(f"  {a.id}  {name}  ({desc}...)")

no agents


## 13. Async usage

Every operation has an async coroutine variant (`.asyncio()`). Pass an `AsyncClient` (same constructor signature) and `await`:

In [12]:
import asyncio

from agent_platform import AsyncClient
from agent_platform.api.sessions import list_sessions
from agent_platform.models import ListSessionsApiV2SessionsGetOwner as ListSessionsOwner


async def main():
    async with AsyncClient(api_key=API_KEY, base_url=BASE_URL) as ac:
        page = await list_sessions.asyncio(client=ac, owner=ListSessionsOwner.ME, page=1, size=5)
        return [s.id for s in page.items]


session_ids = await main()
print(session_ids)

[UUID('fcca1e7f-1e1c-405d-b3d0-57ef9a277005'), UUID('1aeff5dc-dd5f-4b5b-b264-d31a8a00f061'), UUID('f166d05d-19ba-4319-af1a-a73cbba85083'), UUID('44e46cb0-7b85-468b-ae6f-9503e5b5f886'), UUID('63e589f5-c2c6-43df-8321-7312cc7e9942')]


## 14. Error handling

On non-2xx responses, the operation functions raise `UnexpectedStatus` by default. Use `*.sync_detailed()` / `*.asyncio_detailed()` if you want the raw response (status code, headers, parsed body) instead of raising.

Status codes you'll see most often:
- `404` — session/memory/skill/etc. not found (you supplied a bad id)
- `409` — conflict (creating an agent/skill/environment with an identifier that already exists)
- `422` — validation error (Pydantic validation failed on your request body)
- `429` — quota exceeded (check `GET /sessions/quota` for current usage)

In [13]:
from uuid import uuid4

from agent_platform.api.sessions import get_session
from agent_platform.errors import UnexpectedStatus

try:
    get_session.sync(client=client, id=str(uuid4()))  # random id, almost certainly 404
except UnexpectedStatus as exc:
    print(f"got {exc.status_code}: {exc.content!r}")

## What next

- Read [`agent_platform.api`](../src/agent_platform/api/) — every operation has a Python module with `.sync` / `.sync_detailed` / `.asyncio` / `.asyncio_detailed` variants and the canonical kwarg signature.
- Read [`agent_platform.models`](../src/agent_platform/models/) — every request/response shape as a typed Pydantic v2 `BaseModel`.
- The CLI (`agp` command, Phase 3 work) wraps these same calls for terminal use.
- Live drift detection ensures this SDK never diverges from the deployed API — if you see a `404` on an endpoint listed in the docs, run `git pull && pip install -U ./sdk/python` and try again.

Questions? File an issue on `hcompai/agent_platform`.